# 12 — Edge-case tests

Offline, deterministic checks for the failure modes in the Video Assistant pipeline. These tests exercise the same contracts used by `01_audio_processor`, `02_transcriber`, `08_llamaindex_retriever`, and `09_agents`, while replacing network, audio, Whisper, and LLM calls with small fakes.

The expected behavior is deliberate: fail clearly, preserve useful diagnostics, and never invent a transcript answer when the source does not support it.

**Fixed in this version:** four setup/test cells were previously saved as `"cell_type": "markdown"` instead of `"code"` — Jupyter never executes markdown cells, so they silently never ran. The duplicate *code* cells that did run were an earlier, simpler version that dropped the greeting-detection checks entirely. That's why the saved output in `main.ipynb` shows only 5 passing checks with no "Greetings / small talk" line, even though `08_llamaindex_retriever.ipynb` implements greeting handling. This version keeps one clean, executing copy of each cell, with the greeting checks included and asserted for real.

## 1. Shared fakes and helpers

In [ ]:
import re
from dataclasses import dataclass
from pathlib import Path

MIN_TRANSCRIPT_CHARS = 20
NOT_COVERED_MESSAGE = {
    'english': 'I could not find this information in the meeting transcript.',
    'arabic': 'لم أتمكن من العثور على هذه المعلومة في نص الاجتماع.',
}

GREETING_RESPONSES = {
    'english': 'Hello! I am your AI Video Assistant, ready to help you with any questions or details about this recording. How can I help you?',
    'arabic': 'أهلاً وسهلاً بك! أنا المساعد الذكي الخاص بالتسجيلات والاجتماعات، جاهز للإجابة عن أي أسئلة أو تفاصيل تخص محتوى هذا التسجيل. كيف يمكنني مساعدتك؟',
}

GREETING_PATTERNS = [
    r"^(?:أ|ا|إ)?هل(?:ا|اً|ا)?(?:\s+(?:وسهلا|بيك|بك|بيكم))?$",
    r"^مرحب(?:ا|اً)?(?:\s+(?:بيك|بك|بيكم|وسهلا))?$",
    r"^سلام(?:\s+(?:عليكم|عليك))?$",
    r"^السلام\s+عليكم(?:\s+ورحم(?:ة|ه)\s+الله(?:\s+وبركات(?:ة|ه))?)?$",
    r"^ازيك(?:\s+(?:عامل\s+ايه|يا\s+بطل|يا\s+غالي|تمام))?$",
    r"^عامل\s+(?:ايه|إيه)$",
    r"^(?:ازيك\s+)?عامل\s+(?:ايه|إيه)$",
    r"^(?:أ|ا)?خبارك|شخبارك|اخباركم|شخباركم$",
    r"^كيف(?:ك|كم|\s+حالك|\s+حالكم|\s+الحال|\s+الأمور)$",
    r"^صباح\s+(?:الخير|النور|الورد)$",
    r"^مساء\s+(?:الخير|النور|الورد)$",
    r"^شلونك|شلونكم$",
    r"^هاي$",
    r"^هلو$",
    r"^(?:شكرا|شكراً|تسلم|تسلملي|تسلموا|يعطيك\s+العافي(?:ة|ه)|يعطيكم\s+العافي(?:ة|ه))$",
    r"^(?:مين\s+انت|من\s+انت|انت\s+مين|بتعمل\s+ايه)$",
    r"^hi(?:ya)?$",
    r"^hello(?:\s+there)?$",
    r"^hey(?:\s+there)?$",
    r"^greetings$",
    r"^good\s+(?:morning|afternoon|evening|day)$",
    r"^how\s+(?:are\s+you(?:\s+doing)?|is\s+it\s+going|do\s+you\s+do)$",
    r"^what(?:'s|\s+is)\s+up$",
    r"^who\s+(?:are\s+you|made\s+you)$",
    r"^what\s+can\s+you\s+do$",
    r"^(?:thanks|thank\s+you)$",
    r"^nice\s+to\s+meet\s+you$",
]

def normalize_greeting_text(text: str) -> str:
    t = text.strip().lower()
    t = re.sub(r"[\u064b-\u0670]", "", t)
    t = re.sub(r"[\?!,\.،؟!\-_~@#$%^&*()\[\]{}]+", " ", t)
    t = re.sub(r"[إأآا]", "ا", t)
    t = re.sub(r"ة\b", "ه", t)
    t = re.sub(r"ى\b", "ي", t)
    return re.sub(r"\s+", " ", t).strip()

RAW_GREETING_WORDS = {
    "اهلا", "أهلا", "وسهلا", "مرحبا", "سلام", "عليكم", "ورحمة", "ورحمه", "الله", "وبركاته", "وبركته",
    "ازيك", "عامل", "ايه", "إيه", "اخبارك", "أخبارك", "اخباركم", "كيفك", "كيفكم",
    "حالك", "حالكم", "الحال", "صباح", "مساء", "الخير", "النور", "الورد", "هاي",
    "هلو", "شكرا", "تسلم", "تسلملي", "مين", "انت", "إنت", "تمام", "الحمد", "لله",
    "شلونك", "شلونكم", "يا", "بطل", "غالي",
    "hi", "hello", "hey", "how", "are", "you", "good", "morning", "afternoon",
    "evening", "thanks", "thank", "whats", "up", "there", "doing", "fine", "im",
    "i", "am", "who", "what", "can", "do"
}
GREETING_WORDS = {normalize_greeting_text(w) for w in RAW_GREETING_WORDS}

def is_greeting(text: str) -> bool:
    norm = normalize_greeting_text(text)
    if not norm:
        return False
    for pat in GREETING_PATTERNS:
        if re.match(pat, norm):
            return True
    sub_parts = norm.split(" ")
    if len(sub_parts) <= 7 and all(w in GREETING_WORDS for w in sub_parts):
        return True
    return False

def detect_script_mismatch(transcript: str, language: str) -> bool:
    letters = re.findall(r'[^\W\d_]', transcript, flags=re.UNICODE)
    if not letters:
        return False
    arabic_ratio = sum('\u0600' <= ch <= '\u06ff' for ch in letters) / len(letters)
    return arabic_ratio < 0.3 if language == 'arabic' else arabic_ratio > 0.3

def validate_transcript(transcript: str, language: str = 'english') -> dict:
    transcript = transcript or ''
    if len(transcript.strip()) < MIN_TRANSCRIPT_CHARS:
        return {'blocked': True, 'warning': None, 'error': 'Audio is silent, near-silent, or unintelligible.'}
    warning = None
    if detect_script_mismatch(transcript, language):
        warning = f'Transcript may not match selected language: {language}.'
    return {'blocked': False, 'warning': warning, 'error': None}

def process_local_file(path: str) -> list:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f'No such file: {path}')
    if file_path.read_bytes()[:4] != b'RIFF':
        raise RuntimeError(f"Could not read '{path}' as audio — the file may be corrupted or unsupported.")
    return [f'{path}_chunk_0.wav']

def process_youtube_url(url: str) -> list:
    if 'private' in url or 'unavailable' in url or 'broken' in url:
        raise RuntimeError(f"Could not download audio for '{url}'.")
    return ['downloads/video_chunk_0.wav']


## 2. Silent or near-silent audio

In [ ]:
silent_result = validate_transcript('   ', language='english')
garbage_result = validate_transcript('uh uh uh', language='english')
assert silent_result['blocked'] is True
assert garbage_result['blocked'] is True
assert 'silent' in silent_result['error']
print('PASS: empty and near-empty Whisper output is blocked before summarization.')


## 3. Selected language mismatch

In [ ]:
arabic_in_english = validate_transcript(
    'هذا نص عربي طويل بما يكفي لاختبار اكتشاف اللغة.',
    language='english',
)
english_in_arabic = validate_transcript(
    'This is a sufficiently long English transcript for the language check.',
    language='arabic',
)
assert arabic_in_english['blocked'] is False
assert arabic_in_english['warning'] is not None
assert english_in_arabic['blocked'] is False
assert english_in_arabic['warning'] is not None
print('PASS: a likely language mismatch warns without discarding the transcript.')


## 4. Greetings / small talk, and weak RAG retrieval (no hallucination)

In [ ]:
@dataclass
class FakeNode:
    text: str
    score: float

class FakeRetriever:
    def __init__(self, nodes):
        self.nodes = nodes

    def retrieve(self, question):
        return self.nodes

def ask_without_hallucination(retriever, question, language='english', llm=None, threshold=0.35):
    if is_greeting(question):
        return GREETING_RESPONSES.get(language, GREETING_RESPONSES['english'])
    nodes = retriever.retrieve(question)
    best_score = max((node.score for node in nodes), default=0.0)
    if best_score < threshold:
        return NOT_COVERED_MESSAGE.get(language, NOT_COVERED_MESSAGE['english'])
    return llm(question, '\n\n'.join(node.text for node in nodes))

llm_calls = []
def forbidden_llm(question, context):
    llm_calls.append((question, context))
    return 'hallucinated answer'

# Test 1: Greeting is handled politely without LLM hallucination or not-covered message
greeting_answer_en = ask_without_hallucination(
    FakeRetriever([FakeNode('The meeting discussed deployment timing.', 0.08)]),
    'hello how are you',
    language='english',
    llm=forbidden_llm,
)
assert greeting_answer_en == GREETING_RESPONSES['english']

greeting_answer_ar = ask_without_hallucination(
    FakeRetriever([FakeNode('The meeting discussed deployment timing.', 0.08)]),
    'ازيك عامل ايه',
    language='arabic',
    llm=forbidden_llm,
)
assert greeting_answer_ar == GREETING_RESPONSES['arabic']

# Test 2: Unrelated question not covered by transcript returns not-covered message
answer = ask_without_hallucination(
    FakeRetriever([FakeNode('The meeting discussed deployment timing.', 0.08)]),
    'What was the office catering budget?',
    language='english',
    llm=forbidden_llm,
)
assert answer == NOT_COVERED_MESSAGE['english']
assert llm_calls == []
print('PASS: greetings return conversational replies; weak retrieval returns not covered and skips the LLM call.')


## 5. Broken, private, or unavailable YouTube URL

In [ ]:
for status in ('broken', 'private', 'unavailable'):
    try:
        process_youtube_url(f'https://youtube.example/{status}')
    except RuntimeError as error:
        assert 'Could not download audio' in str(error)
    else:
        raise AssertionError(f'{status} URL should fail cleanly')
print('PASS: unavailable YouTube sources become readable processing errors.')


## 6. Unsupported or corrupted local audio

In [ ]:
bad_path = Path('edge_case_corrupt_audio.bin')
bad_path.write_bytes(b'not an audio file')
try:
    process_local_file(str(bad_path))
except RuntimeError as error:
    assert 'corrupted or unsupported' in str(error)
finally:
    bad_path.unlink(missing_ok=True)
print('PASS: corrupted or unsupported local files fail with a normalized error.')


## Summary

In [ ]:
summary = [
    ('Silent / near-silent audio', 'blocked before LLM work'),
    ('Language mismatch', 'warning, transcript retained'),
    ('Greetings / small talk', 'conversational response, LLM skipped'),
    ('No relevant RAG chunk', 'not covered, LLM skipped'),
    ('Broken/private YouTube URL', 'readable processing error'),
    ('Corrupt local audio', 'readable processing error'),
]
for case, behavior in summary:
    print(f'{case}: {behavior}')
print('\nAll edge-case checks passed.')
